# Graph Neural Networks (GNN) — Theory & Mathematical Foundations

This notebook covers the theoretical background of GNNs applied to combinatorial
optimisation. For the code, training, and benchmarks see **`notebook_implementation.ipynb`**.


## 1. Definition

**Graph Neural Networks (GNNs)** are a family of deep learning models designed to operate directly on graph-structured data. Rather than assuming a fixed Euclidean input layout (like images or sequences), GNNs learn representations by propagating and aggregating information along the edges of a graph.

### Origin

| Era | Contribution |
|-----|-------------|
| 2005–2009 | Gori et al. and Scarselli et al. — first GNN formulation using fixed-point iterations |
| 2017 | Kipf & Welling — Graph Convolutional Networks (GCN), spectral approach |
| 2017 | Hamilton et al. — GraphSAGE, inductive node embeddings via neighborhood sampling |
| 2018 | Veličković et al. — Graph Attention Networks (GAT), attention-weighted aggregation |
| 2019 | Joshi et al. — GCN applied to TSP edge prediction |

### Category

GNNs belong to **geometric deep learning** — a generalization of CNNs to non-Euclidean domains. For combinatorial optimization, they are used as **construction heuristics** (edge scoring → tour decoding) or **improvement heuristics** (learned local search).

### Key Concepts

- **Node embedding** $h_v^{(l)}$: a vector representation of node $v$ at layer $l$, encoding local neighborhood structure.
- **Message passing**: nodes iteratively aggregate features from their neighbors.
- **Readout / pooling**: global graph-level or edge-level prediction from node embeddings.
- **Permutation invariance**: the output is invariant to node ordering, which is natural for combinatorial problems.

## 2. Formal Description

### Graph Representation

A weighted graph is defined as $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathbf{X}, \mathbf{E})$ where:

- $\mathcal{V} = \{v_1, \dots, v_n\}$ — set of $n$ nodes (cities)
- $\mathcal{E} \subseteq \mathcal{V} \times \mathcal{V}$ — set of edges
- $\mathbf{X} \in \mathbb{R}^{n \times d_x}$ — node feature matrix; row $\mathbf{x}_i$ encodes features of node $i$ (e.g. 2D coordinates)
- $\mathbf{E}_{ij} \in \mathbb{R}^{d_e}$ — edge feature vector (e.g. Euclidean distance $\|x_i - x_j\|_2$)
- $\mathbf{A} \in \{0,1\}^{n \times n}$ — adjacency matrix; $A_{ij} = 1$ iff $(i,j) \in \mathcal{E}$

For TSP, the graph is **complete** ($|\mathcal{E}| = n(n-1)/2$) and **undirected**.

---

### Message Passing Neural Network (MPNN) Framework

The general message-passing update at layer $l$ is:

$$\mathbf{m}_v^{(l)} = \bigoplus_{u \in \mathcal{N}(v)} \phi^{(l)}\!\left(\mathbf{h}_v^{(l-1)},\, \mathbf{h}_u^{(l-1)},\, \mathbf{e}_{uv}\right)$$

where:
- $\mathbf{m}_v^{(l)}$ — aggregated message at node $v$, layer $l$
- $\bigoplus$ — permutation-invariant aggregation (sum, mean, or max)
- $\mathcal{N}(v)$ — neighbourhood of node $v$
- $\phi^{(l)}$ — **message function** at layer $l$; computes a per-edge signal (e.g. MLP)
- $\mathbf{h}_v^{(l-1)}, \mathbf{h}_u^{(l-1)}$ — node embeddings of centre and neighbour from the previous layer
- $\mathbf{e}_{uv}$ — static edge feature vector between $u$ and $v$

$$\mathbf{h}_v^{(l)} = \psi^{(l)}\!\left(\mathbf{h}_v^{(l-1)},\, \mathbf{m}_v^{(l)}\right)$$

where:
- $\mathbf{h}_v^{(l)}$ — updated node embedding at layer $l$
- $\psi^{(l)}$ — **update function** at layer $l$; fuses the previous embedding with the aggregated message (e.g. GRU or MLP)
- $\mathbf{h}_v^{(0)} = \mathbf{x}_v$ — initial node feature

---

### GCN Variant (Kipf & Welling, 2017)

The simplest spectral formulation:

$$\mathbf{H}^{(l+1)} = \sigma\!\left(\tilde{\mathbf{D}}^{-\frac{1}{2}}\,\tilde{\mathbf{A}}\,\tilde{\mathbf{D}}^{-\frac{1}{2}}\,\mathbf{H}^{(l)}\,\mathbf{W}^{(l)}\right)$$

where:
- $\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}_n$ — adjacency matrix with self-loops added
- $\tilde{\mathbf{D}}_{ii} = \sum_j \tilde{A}_{ij}$ — diagonal degree matrix of $\tilde{\mathbf{A}}$
- $\tilde{\mathbf{D}}^{-1/2}\tilde{\mathbf{A}}\tilde{\mathbf{D}}^{-1/2}$ — symmetric normalisation; prevents embeddings from scaling with node degree
- $\mathbf{W}^{(l)} \in \mathbb{R}^{d^{(l)} \times d^{(l+1)}}$ — learnable weight matrix at layer $l$
- $\sigma$ — non-linearity (ReLU)

---

### TSP-GNN: Edge Prediction Objective (Joshi et al., 2019)

For TSP, the model is trained to predict whether each edge $(i,j)$ belongs to the optimal tour. Let $\hat{p}_{ij} \in [0,1]$ be the predicted edge probability.

**Edge embedding update:**

$$\mathbf{e}_{ij}^{(l+1)} = \text{BN}\!\left(\mathbf{W}_1^{(l)}\,\mathbf{h}_i^{(l)} + \mathbf{W}_2^{(l)}\,\mathbf{h}_j^{(l)} + \mathbf{W}_3^{(l)}\,\mathbf{e}_{ij}^{(l)}\right)$$

where:
- $\mathbf{e}_{ij}^{(l+1)}$ — updated edge embedding at layer $l+1$
- $\mathbf{W}_1^{(l)}, \mathbf{W}_2^{(l)}$ — learnable weights applied to the source and target node embeddings
- $\mathbf{W}_3^{(l)}$ — learnable weight on the previous edge embedding (carries forward prior edge state)
- $\text{BN}$ — Batch Normalisation; stabilises activations across the mini-batch

**Node embedding update** (aggregating over all neighbors):

$$\mathbf{h}_i^{(l+1)} = \mathbf{h}_i^{(l)} + \text{ReLU}\!\left(\text{BN}\!\left(\mathbf{W}_4^{(l)}\,\mathbf{h}_i^{(l)} + \sum_{j \in \mathcal{N}(i)} \hat{\eta}_{ij}^{(l)}\,\mathbf{W}_5^{(l)}\,\mathbf{e}_{ij}^{(l+1)}\right)\right)$$

where:
- $\mathbf{h}_i^{(l)}$ — residual skip connection; carries the previous node embedding unchanged, preserving gradient flow
- $\mathbf{W}_4^{(l)}$ — learnable self-projection on the current node embedding
- $\hat{\eta}_{ij}^{(l)} = \text{softmax}_j\!\left(\mathbf{e}_{ij}^{(l+1)}\right)$ — attention weights over the edges leaving $i$; normalised so $\sum_j \hat{\eta}_{ij}^{(l)} = 1$
- $\mathbf{W}_5^{(l)}$ — learnable projection applied to edge embeddings before aggregation
- $\sum_{j \in \mathcal{N}(i)}$ — attention-weighted sum over all neighbours of node $i$

**Output layer:**

$$\hat{p}_{ij} = \sigma\!\left(\mathbf{w}^\top \mathbf{e}_{ij}^{(L)}\right), \quad \sigma(x) = \frac{1}{1+e^{-x}}$$

where:
- $\hat{p}_{ij} \in (0,1)$ — predicted probability that edge $(i,j)$ belongs to the optimal tour
- $\mathbf{w}$ — learnable output weight vector (linear classifier on the final edge embedding)
- $\mathbf{e}_{ij}^{(L)}$ — final edge embedding after $L$ GNN layers
- $\sigma$ — sigmoid function; maps any real logit to a probability in $(0,1)$

**Training loss** (binary cross-entropy over all edges):

$$\mathcal{L} = -\frac{1}{|\mathcal{E}|} \sum_{(i,j) \in \mathcal{E}} \left[y_{ij}\,\log \hat{p}_{ij} + (1 - y_{ij})\,\log(1 - \hat{p}_{ij})\right]$$

where:
- $|\mathcal{E}|$ — total number of edges; $n(n-1)/2$ for a complete undirected graph
- $y_{ij} \in \{0,1\}$ — ground-truth label: 1 if edge $(i,j)$ is in the reference tour, 0 otherwise
- $y_{ij}\log\hat{p}_{ij}$ — tour-edge term; penalises low probability on true tour edges
- $(1-y_{ij})\log(1-\hat{p}_{ij})$ — non-edge term; penalises high probability on non-tour edges

## 3. Architecture / Algorithm Steps

### Overall Pipeline

```
Input graph G                  Edge probabilities              TSP tour
 (node coords)   →  GNN encoder  →  p̂_ij ∈ [0,1]   →  Decoder  →  π
```

---

### Step 1 — Input Encoding

Each node $i$ is embedded into a $d$-dimensional space:

$$\mathbf{h}_i^{(0)} = \mathbf{W}_{\text{in}}\,\mathbf{x}_i + \mathbf{b}_{\text{in}}, \quad \mathbf{W}_{\text{in}} \in \mathbb{R}^{d \times d_x}$$

where:
- $\mathbf{h}_i^{(0)}$ — initial node embedding for city $i$
- $\mathbf{W}_{\text{in}}$ — learnable weight matrix; projects raw node features into the model dimension $d$
- $\mathbf{x}_i$ — raw node feature vector (e.g. 2-D coordinates $(x,y)$ normalised to $[0,1]$)
- $d_x$ — input feature dimension ($d_x = 2$ for plain TSP; $d_x = 5$ for TSPTW-D)
- $d$ — hidden dimension (64 / 128 / 256 for small / medium / large preset)

Each edge $(i,j)$ is initialized with its Euclidean distance:

$$\mathbf{e}_{ij}^{(0)} = \mathbf{W}_{\text{edge}}\,\left[\,\|x_i - x_j\|_2\,\right] + \mathbf{b}_{\text{edge}}$$

where:
- $\mathbf{e}_{ij}^{(0)}$ — initial edge embedding for the pair $(i,j)$
- $\mathbf{W}_{\text{edge}}$ — learnable projection from scalar edge features to dimension $d$
- $\|x_i - x_j\|_2$ — Euclidean distance between cities $i$ and $j$ (scalar input feature)

---

### Step 2 — $L$ Graph Convolutional Layers

For $l = 1, \dots, L$, repeat:

**2a. Edge update** — combines incident node embeddings and previous edge state:

$$\mathbf{e}_{ij}^{(l)} = \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_1^{(l)}\mathbf{h}_i^{(l-1)} + \mathbf{W}_2^{(l)}\mathbf{h}_j^{(l-1)} + \mathbf{W}_3^{(l)}\mathbf{e}_{ij}^{(l-1)}\right)\right)$$

where:
- $\mathbf{W}_1^{(l)}, \mathbf{W}_2^{(l)}$ — learnable weights applied to source and target node embeddings
- $\mathbf{W}_3^{(l)}$ — learnable weight on the previous edge embedding
- $\text{LN}$ — Layer Normalisation; stabilises activations per sample (used instead of BN in this implementation)
- $\text{ReLU}$ — rectified linear unit $\max(0, \cdot)$; introduces non-linearity

**2b. Attention gate** — soft selection over neighboring edges:

$$\hat{\eta}_{ij}^{(l)} = \frac{\exp\!\left(\mathbf{e}_{ij}^{(l)}\right)}{\sum_{k \in \mathcal{N}(i)} \exp\!\left(\mathbf{e}_{ik}^{(l)}\right)}$$

where:
- $\hat{\eta}_{ij}^{(l)}$ — attention weight on edge $(i,j)$; normalised over all outgoing edges of node $i$
- $\exp(\mathbf{e}_{ij}^{(l)})$ — element-wise exponential of the updated edge embedding
- $\mathcal{N}(i)$ — all neighbours of node $i$ (complete graph: every other city)
- $\sum_k$ — normalisation sum; ensures $\sum_j \hat{\eta}_{ij}^{(l)} = \mathbf{1}$ (soft-max)

**2c. Node update** — residual aggregation:

$$\mathbf{h}_i^{(l)} = \mathbf{h}_i^{(l-1)} + \text{ReLU}\!\left(\text{LN}\!\left(\mathbf{W}_4^{(l)}\mathbf{h}_i^{(l-1)} + \sum_{j \in \mathcal{N}(i)} \hat{\eta}_{ij}^{(l)} \cdot \mathbf{W}_5^{(l)} \mathbf{e}_{ij}^{(l)}\right)\right)$$

where:
- $\mathbf{h}_i^{(l-1)}$ — residual skip connection; preserves gradient flow across layers (analogous to ResNet)
- $\mathbf{W}_4^{(l)}$ — learnable self-projection on the previous node embedding
- $\hat{\eta}_{ij}^{(l)}$ — attention weight (from step 2b); weights the contribution of each neighbour
- $\mathbf{W}_5^{(l)}$ — learnable projection applied to edge embeddings before aggregation
- $\sum_{j \in \mathcal{N}(i)}$ — attention-weighted sum over all neighbours of node $i$

---

### Step 3 — Edge Scoring Head

After $L$ layers, a linear classifier produces edge logits:

$$\hat{p}_{ij} = \text{sigmoid}\!\left(\text{MLP}\!\left(\mathbf{e}_{ij}^{(L)}\right)\right) \in [0,1]$$

where:
- $\hat{p}_{ij}$ — predicted probability that edge $(i,j)$ belongs to the optimal tour
- $\text{MLP}$ — two-layer perceptron (linear → ReLU → linear); maps $d$-dimensional edge embedding to a scalar logit
- $\mathbf{e}_{ij}^{(L)}$ — final edge embedding after $L$ GNN layers
- $\text{sigmoid}$ — maps the logit to a probability in $(0,1)$

---

### Step 4 — Tour Decoding

The edge scores $\hat{p}_{ij}$ are used to guide a combinatorial decoder. Two common strategies:

| Decoder | Description |
|---------|-------------|
| **Greedy** | At each step, extend the partial tour with the highest-scoring unvisited neighbor |
| **Beam search** | Maintain $k$ partial tours simultaneously; select top-$k$ by cumulative log-prob |
| **MCTS** | Monte Carlo Tree Search guided by $\hat{p}_{ij}$ as prior probabilities |

The final output is a permutation $\pi = (\pi_1, \dots, \pi_n)$ visiting all cities exactly once.

---

### Training Procedure (Supervised)

```
For each epoch:
  For each instance (G, π*) in training set:
    1. Forward pass: compute ĥ, ê through L GNN layers
    2. Compute edge probabilities p̂_ij
    3. Compute loss L = BCE(p̂, y*)   where y*_ij = 1 if (i,j) ∈ π*
    4. Backprop through all layers (via chain rule)
    5. Update weights with Adam optimizer
```

**Alternatively**, reinforcement learning (REINFORCE) is used when optimal labels $\pi^*$ are unavailable:

$$\nabla_\theta \mathcal{J}(\theta) = \mathbb{E}_{\pi \sim p_\theta}\!\left[(L(\pi) - b)\,\nabla_\theta \log p_\theta(\pi)\right]$$

where:
- $\theta$ — learnable parameters of the GNN
- $\mathcal{J}(\theta)$ — expected tour length under the model's distribution
- $L(\pi) = \sum_{i} d(\pi_i, \pi_{i+1})$ — total length of sampled tour $\pi$
- $b$ — baseline value (e.g. greedy rollout length); reduces variance of the gradient estimate
- $L(\pi) - b$ — advantage: positive if the sampled tour is worse than baseline, negative if better
- $\nabla_\theta \log p_\theta(\pi)$ — policy gradient; steers the model toward tours with lower cost

## 4. Complexity Analysis

### Notation

| Symbol | Meaning |
|--------|---------|
| $n$ | number of nodes (cities) |
| $\|\mathcal{E}\|$ | number of edges |
| $L$ | number of GNN layers |
| $d$ | embedding dimension |
| $k$ | beam width (decoder) |

For a **complete graph** (TSP), $|\mathcal{E}| = \dfrac{n(n-1)}{2} = O(n^2)$.

---

### Time Complexity

**Per GNN layer:**

Each edge update requires $O(d^2)$ operations (matrix-vector multiply), repeated for all edges:

$$T_{\text{edge}}^{(l)} = O(|\mathcal{E}| \cdot d^2) = O(n^2 d^2)$$

Each node update aggregates over all neighbors (degree $n-1$ in complete graph):

$$T_{\text{node}}^{(l)} = O(n \cdot (n-1) \cdot d) = O(n^2 d)$$

**Full encoder ($L$ layers):**

$$T_{\text{encoder}} = O(L \cdot n^2 \cdot d^2)$$

**Greedy decoder:** $O(n^2)$ — one pass over the edge score matrix.

**Beam search decoder:** $O(k \cdot n^2)$.

**Total (inference):**

$$\boxed{T_{\text{total}} = O(L \cdot n^2 \cdot d^2)}$$

---

### Space Complexity

| Component | Memory |
|-----------|--------|
| Node embeddings $\mathbf{H}^{(l)}$ | $O(L \cdot n \cdot d)$ |
| Edge embeddings $\mathbf{E}^{(l)}$ | $O(L \cdot n^2 \cdot d)$ |
| Weight matrices (all layers) | $O(L \cdot d^2)$ |
| Adjacency / score matrix | $O(n^2)$ |

Dominant term: **edge embeddings** $O(L \cdot n^2 \cdot d)$, which becomes prohibitive for large $n$.

$$\boxed{S_{\text{total}} = O(L \cdot n^2 \cdot d)}$$

---

### Comparison with Other Methods

| Method | Time (inference) | Scales to $n = 10^4$? |
|--------|-----------------|----------------------|
| Exact (Concorde) | $O(2^n)$ worst case | No |
| LKH-3 heuristic | $O(n^2 \log n)$ per run | Barely |
| Transformer (Kool 2019) | $O(n^2 d)$ | Partially |
| **GNN (Joshi 2019)** | $O(L n^2 d^2)$ | No (memory bound) |
| Sparse GNN | $O(L \cdot k_{\text{sp}} \cdot n \cdot d^2)$ | Yes (with sparse $k$-NN graph) |

> **Note:** the $O(n^2)$ bottleneck can be reduced by working on a $k$-nearest-neighbor sparse graph instead of the complete graph, bringing complexity down to $O(L \cdot k \cdot n \cdot d^2)$.

## 5. Strengths and Limitations

### Strengths

| Property | Detail |
|----------|--------|
| **Graph-native** | Directly encodes the combinatorial structure (nodes = cities, edges = connections) without flattening or padding |
| **Permutation invariance** | Output is invariant to node relabeling — the model does not depend on an arbitrary ordering |
| **Edge-level supervision** | Predicts *which edges* belong to the tour, providing fine-grained signal that generalizes well across instances |
| **Fast inference** | Once trained, a forward pass produces edge scores in $O(L n^2 d^2)$ — milliseconds for $n \leq 100$ |
| **Composable** | GNN encoder can be combined with any downstream decoder (greedy, beam search, MCTS) |
| **Inductive** | Can generalize to graph sizes not seen during training (with some degradation) |

---

### Limitations

| Limitation | Impact |
|------------|--------|
| **Quadratic memory** in $n$ | Edge embeddings require $O(n^2 d)$ memory; infeasible for $n \gtrsim 1000$ without sparse approximation |
| **Requires training data** | Supervised variant needs optimal (or near-optimal) tours from a solver — expensive to generate at scale |
| **Optimality gap** | Typically 1–5 % above optimal for $n \leq 100$; gap grows with $n$ |
| **Generalization degradation** | Models trained on $n = 50$ perform poorly on $n = 500$ without retraining or fine-tuning |
| **Decoder sensitivity** | Final tour quality depends heavily on the decoding strategy; a weak decoder wastes a strong encoder |
| **No hard constraint enforcement** | The model may produce invalid tours (revisited cities); post-processing needed |

## 6. Use Case Explanation

### When to Use a GNN for Combinatorial Optimization

GNNs are well-suited when:

1. **The problem is naturally graph-structured** — cities, depots, and routes form a graph. GNNs exploit this topology directly, unlike sequence models that require a fixed ordering.

2. **Repeated solving on similar instances is needed** — the training cost is amortized over thousands of inferences. If you need to solve the same problem distribution millions of times (e.g. logistics routing), the one-time training cost is negligible.

3. **Near-real-time decisions are required** — classical solvers (Concorde, LKH) take seconds to minutes per instance; a trained GNN runs in milliseconds.

4. **$n \leq 200$** — GNNs achieve competitive quality in this regime. Beyond $n \approx 500$, memory and generalization issues become dominant.

---

### Why GNNs Specifically (vs. Transformers or Pointer Nets)

| Model | Inductive bias | TSP quality ($n=100$) |
|-------|---------------|----------------------|
| Pointer Network | Sequential, attention over nodes | ~5 % gap |
| Transformer (Kool) | Global pairwise attention | ~1–2 % gap |
| **GNN (Joshi)** | Local message passing + edge features | ~1–3 % gap |

GNNs have a **stronger structural prior**: they explicitly model edges (not just nodes), which aligns with the TSP objective — selecting a subset of edges forming a Hamiltonian cycle. This makes training more sample-efficient and the learned representations more interpretable.

---

### Position in the Optimization Pipeline

```
Problem instance
       │
       ▼
  GNN encoder          ← learns edge likelihoods from graph structure
       │
       ▼
  Edge scores p̂_ij
       │
       ├──► Greedy decoder     (fast, ~2–5 % gap)
       ├──► Beam search        (better quality, O(k·n²))
       └──► Local search (2-opt, LKH)  ← use GNN output as warm start
```

The GNN is most valuable as a **heuristic initializer**: its edge scores prune the search space for classical solvers, enabling LKH to find near-optimal solutions much faster than from scratch.

## 10. Experimental Analysis

### Scalability

The benchmark confirms the **$O(n^2)$ bottleneck** predicted by complexity analysis. The dominant cost is the edge embedding tensor $\mathbf{E}^{(l)} \in \mathbb{R}^{n \times n \times d}$, which must be allocated and updated at every layer.

Two mitigations exist:
- **Sparse graph**: restrict edges to $k$ nearest neighbors. Reduces memory to $O(k \cdot n \cdot d)$, but may cut important long-range edges.
- **Hierarchical GNN**: cluster cities into groups, run GNN within clusters, then across clusters. Reduces effective $n$ at each level.

---

### Behavioral Analysis

After training, the attention weights $\hat{\eta}_{ij}^{(l)}$ (edge attention gate in `GNNLayer`) can be visualized to understand what the model has learned:

- **Early layers** ($l=1,2$): high attention on geometrically close neighbors — the model learns local proximity.
- **Later layers** ($l=3,4$): attention becomes more selective, focusing on edges that are globally consistent with a low-cost tour (e.g. edges that don't "cross" other high-score edges).

This mirrors what a human solver does: first identify candidate edges locally, then filter for global consistency.

---

### Comparison with Other Models

| Model | Architecture | Training | TSP $n=100$ gap | Memory |
|-------|-------------|----------|-----------------|--------|
| **Pointer Network** (Vinyals 2015) | Seq2Seq + attention | RL (REINFORCE) | ~5 % | $O(n \cdot d)$ |
| **Transformer AM** (Kool 2019) | Encoder-decoder, multi-head attention | RL (REINFORCE + greedy baseline) | ~1–2 % | $O(n^2 \cdot d)$ |
| **GNN** (Joshi 2019) | Residual GCN, edge embeddings | Supervised (BCE) | ~1–3 % | $O(L \cdot n^2 \cdot d)$ |
| **LKH-3** (Helsgott) | Metaheuristic | None (exact search) | < 0.5 % | $O(n^2)$ |

Key takeaway: the GNN is **competitive in quality** with Transformer-based models, but its $O(n^2)$ memory is a hard constraint that limits practical use to $n \leq 200$–$300$.

---

### Limitations

1. **No constraint handling** — the model produces edge probabilities, not a valid tour. The greedy decoder may revisit cities or violate time windows (TSPTW). Post-processing (repair heuristic) is needed.
2. **Supervised training requires labels** — for $n > 10$, optimal labels must come from an external solver (LKH), which is slow and limits dataset size.
3. **Generalization across sizes** — a model trained on $n=50$ degrades significantly on $n=200$. Size-invariant training (mixing instance sizes) partially addresses this.

---

### Improvement Proposals

| Proposal | Expected gain |
|----------|---------------|
| Switch to **RL training** (REINFORCE) | Removes need for optimal labels; scalable to $n > 100$ |
| Use **sparse $k$-NN graph** | Reduces memory from $O(n^2 d)$ to $O(k n d)$; enables $n > 500$ |
| Add **TSPTW node features** $(x, y, e_i, l_i, s_i)$ | Extends the model to time-windowed instances |
| Add **feasibility mask** in decoder | Prevents visiting cities outside their time window |
| **2-opt post-processing** | Improves tour quality by ~1–2 % at low cost |


## 11. References

| Authors | Year | Title | Venue |
|---------|------|-------|-------|
| Scarselli et al. | 2009 | The Graph Neural Network Model | IEEE Trans. Neural Netw. |
| Kipf & Welling | 2017 | Semi-Supervised Classification with Graph Convolutional Networks | ICLR |
| Veličković et al. | 2018 | Graph Attention Networks (GAT) | ICLR |
| Vinyals et al. | 2015 | Pointer Networks | NeurIPS |
| Bello et al. | 2016 | Neural Combinatorial Optimization with Reinforcement Learning | NeurIPS Workshop |
| Kool et al. | 2019 | Attention, Learn to Solve Routing Problems! (Attention Model) | ICLR |
| Joshi et al. | 2019 | An Efficient Graph Convolutional Network Technique for the TSP | INFORMS |
| Solomon | 1987 | Algorithms for the Vehicle Routing and Scheduling Problems with Time Window Constraints | Operations Research |
